In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose

In [28]:
#get the dataframe 
#import the data from a csv-file
df = pd.read_csv('data/eda.csv') #, parse_dates = ['date']
df.shape # (21420, 21)
#make the dates pretty
df['date'] = pd.to_datetime(df.date, format='%Y-%m-%d')
#df['yr_built'] = pd.to_datetime(df.yr_built, format='%Y')
df = df.drop(columns='id.1')
dfc = df.copy()
#do the recasts: 
dfc = df.astype({'price' : 'int64'})
dfc = df.astype({'view':'Int64'})
#dfc = dfc.astype({'sqft_living' : int})
#dfc = dfc.astype({'sqft_lot' : int})
dfc = dfc.astype({'waterfront': 'Int64'})
dfc = dfc.fillna({'waterfront':2})

## now with the renovation: 
dfc['yr_renovated'] = dfc['yr_renovated'].astype(str)
def fix_year_format(year_str):
    if year_str != '0' and len(year_str) == 7:
        return year_str[:-3] 
    return year_str

dfc['yr_renovated'] = dfc['yr_renovated'].apply(fix_year_format)
dfc['yr_renovated'] = pd.to_numeric(
    dfc['yr_renovated'],
    errors='coerce'
)
dfc['yr_renovated'] = dfc['yr_renovated'].astype('Int64')
dfc['yr_renovated'] = dfc['yr_renovated'].fillna(0)


In [29]:

#Enrich the dataset
#make renovated categorical
conditions = [ 
    (dfc['yr_renovated'] == 0),
    (dfc['yr_renovated'] < 1950),
    (dfc['yr_renovated'] >=1950) & ((dfc['yr_renovated'] < 1970)),
    (dfc['yr_renovated'] >=1970) & ((dfc['yr_renovated'] < 1990)),
    (dfc['yr_renovated'] >=1990) & ((dfc['yr_renovated'] < 2010)),
    (dfc['yr_renovated'] >=2010) & ((dfc['yr_renovated'] < 2014)),
    (dfc['yr_renovated'] >=2014 )
]
choices = [
    '0_unrenovated',
    '1_before_fifties',
    '2_before_seventies', 
    '3_before_nineties', #
    '4_before_twotens',
    '5_before_twoforteen',
    '6_recently'
]
dfc['renovation_status'] = np.select(conditions, choices, default='error')

dfc[['yr_renovated', 'renovation_status']]



,yr_renovated,renovation_status
0,0,0_unrenovated
1,1991,4_before_twotens
2,0,0_unrenovated
3,0,0_unrenovated
4,0,0_unrenovated
...,...,...
21415,0,0_unrenovated
21416,0,0_unrenovated
21417,0,0_unrenovated
21418,0,0_unrenovated


In [30]:
#enrich the dataset
#create area coding
counts = dfc['zipcode'].value_counts()
#create coding by "demographics"
area_coding = {}
for zip, count in counts.items():

    if (count < 50):
        area_coding[zip] = '0_deserted'
    elif count >= 50 and count < 150:
        area_coding[zip] = '1_low'
    elif count >= 150 and count < 350:
        area_coding[zip] = '2_medium'
    elif count >= 350 and count < 500:
        area_coding[zip] = '3_lively'
    elif count > 500:
        area_coding[zip] = '4_crowded'
    else:
        print('something went wrong')
#write it into the dataframe 
dfc['area_quality'] = dfc['zipcode']
dfc['area_quality'] = dfc['area_quality'].map(area_coding)

In [31]:
#enrich the dataset
# add price per sqft
dfc['price_per_sqft'] = dfc['price']
for index, value in dfc['price_per_sqft'].items():
    price = (dfc.loc[index, 'price'] // dfc.loc[index, 'sqft_living'])
    dfc.loc[index, 'price_per_sqft'] = price



Requirements: 
* waterfront
* grade of 10 or above (luxury!) (8 or above is good)
* view is 3 or above 
* bedrooms between 3 and 8
* (renovated (there my insight maybe)) * ! recently! renovated OR recently built 
* ok fair - limit the zipcodes to "not crowded" and "not empty" 

In [ ]:
#filter 
dfcf = dfc.copy()

dfcf = dfcf.query('waterfront == 1 and grade >=8') #leaves 111
dfcf = dfcf.query('bedrooms >= 3 and bedrooms <= 11') #leaves 98
dfcf = dfcf.query('view >= 3') #we're at 92 properties 
dfcf = dfcf.query('condition >=3') # just to be safe, is included in grade, leaves the same 
dfcf = dfcf.query('area_quality != "0_deserted" and area_quality != "4_crowded"') #leaves 83
dfcf = dfcf.query('(yr_renovated != 0 and yr_renovated > 1999) or yr_built > 2000') #leaves 17, works for me 
#dfcf = dfcf.query('renovation_status == "6_recently" or renovation_status== "5_twoforteen" or renovation_status == "4_twotens" or yr_built > 2000') #ok now we're talking
# #ok, 22 left 

dfcf.shape

(17, 25)

In [47]:
dfcf.head(22)

,date,price,house_id,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,...,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,renovation_status,area_quality,price_per_sqft
299,2014-06-24,3080000.0,3225069065,301,4.0,5.00,4550.0,18641.0,1.0,1,...,2002,0,98074,47.6053,-122.077,4550.0,19508.0,0_unrenovated,3_lively,676.0
2601,2014-08-15,4500000.0,7738500731,2625,5.0,5.50,6640.0,40014.0,2.0,1,...,2004,0,98155,47.7493,-122.280,3030.0,23408.0,0_unrenovated,3_lively,677.0
2838,2014-11-18,3600000.0,4114601570,2863,3.0,3.25,5020.0,12431.0,2.0,1,...,1941,2002,98144,47.5925,-122.287,3680.0,12620.0,4_before_twotens,2_medium,717.0
3897,2014-07-23,1380000.0,1269200229,3931,3.0,3.25,3786.0,38038.0,1.0,1,...,1978,2006,98070,47.3907,-122.448,2850.0,33361.0,4_before_twotens,1_low,364.0
3975,2014-05-09,2400000.0,724069059,4010,3.0,2.25,3000.0,11665.0,1.5,1,...,2001,0,98075,47.5884,-122.086,3000.0,15959.0,0_unrenovated,3_lively,800.0
4446,2014-05-24,2000000.0,1724069059,4483,5.0,4.00,4580.0,4443.0,3.0,1,...,2004,0,98075,47.5682,-122.059,2710.0,4443.0,0_unrenovated,3_lively,436.0
4722,2014-08-14,1850000.0,9201300050,4759,5.0,2.25,2800.0,8442.0,2.0,1,...,1963,2001,98075,47.5784,-122.076,3220.0,9156.0,4_before_twotens,3_lively,660.0
6179,2014-06-19,2200000.0,2024069008,6228,5.0,4.75,5990.0,10450.0,2.0,1,...,2002,0,98027,47.5554,-122.077,3330.0,14810.0,0_unrenovated,3_lively,367.0
7921,2014-06-23,3400000.0,9362000040,7983,3.0,4.50,5230.0,17826.0,2.0,1,...,2005,0,98040,47.5348,-122.243,3670.0,17826.0,0_unrenovated,2_medium,650.0
8023,2014-06-17,4670000.0,1924059029,8086,5.0,6.75,9640.0,13068.0,1.0,1,...,1983,2009,98040,47.5570,-122.210,3270.0,10454.0,4_before_twotens,2_medium,484.0
